In [37]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import  mean_absolute_error, r2_score, root_mean_squared_error

# Exploration du dataset
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

print("Description du dataset :\n", diabetes.DESCR[:500], "...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)


# Normalisation des données
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# KNN sans optimisation du choix de k
rnn = RadiusNeighborsRegressor(
    radius=3.80,
    weights='distance',
    algorithm='auto',
    metric='minkowski',
    p=2
)
rnn.fit(X_train, y_train)


Description du dataset :
 .. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measur ...


RadiusNeighborsRegressor(radius=3.8, weights='distance')

In [38]:
# Evaluation du modèle
y_pred_train = rnn.predict(X_train)
y_pred_test = rnn.predict(X_test)

print("Nombre de NaN :", np.sum(np.isnan(y_pred_test)))

Nombre de NaN : 0


In [39]:



train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_RMSE = root_mean_squared_error(y_train, y_pred_train)
test_RMSE = root_mean_squared_error(y_test, y_pred_test)
train_MAE = mean_absolute_error(y_train, y_pred_train)
test_MAE = mean_absolute_error(y_test, y_pred_test)

performance_table = pd.DataFrame({
    'Métriques': ['R² Score', 'RMSE', 'MAE'],
    'Entraînement': [train_r2, train_RMSE, train_MAE],
    'Test': [test_r2, test_RMSE, test_MAE]
})

print("Performance du modèle KNN sans optimisation :")
print(performance_table)

Performance du modèle KNN sans optimisation :
  Métriques  Entraînement       Test
0  R² Score           1.0   0.399937
1      RMSE           0.0  57.603501
2       MAE           0.0  47.733904


In [40]:
# Recherche du meilleur rayon et poids
param_grid = {
    'radius': np.arange(0.1, 5.0, 0.1),          
    'weights': ['uniform', 'distance']
}

rnn = RadiusNeighborsRegressor()
grid_search = GridSearchCV(
    rnn, param_grid, cv=7, scoring='r2', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print("\n--- Meilleurs paramètres ---")
print("Meilleur rayon et weights :", grid_search.best_params_)
print("Meilleur R² (CV)      :", grid_search.best_score_.round(3))

Fitting 7 folds for each of 98 candidates, totalling 686 fits

--- Meilleurs paramètres ---
Meilleur rayon et weights : {'radius': np.float64(3.3000000000000003), 'weights': 'distance'}
Meilleur R² (CV)      : 0.398


C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan 0.39262501 0.39779911
 0.37751414 0.38457188 

In [41]:
# Modèle optimisé
best_rnn = grid_search.best_estimator_
best_rnn.fit(X_train, y_train)

# Ensemble de validation pour l'optimisation des hyperparamètres
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42
)


In [42]:
# Evaluation du modèle
y_pred_train = best_rnn.predict(X_train)
y_pred_test = best_rnn.predict(X_test)


train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_RMSE = root_mean_squared_error(y_train, y_pred_train)
test_RMSE = root_mean_squared_error(y_test, y_pred_test)
train_MAE = mean_absolute_error(y_train, y_pred_train)
test_MAE = mean_absolute_error(y_test, y_pred_test)

performance_table = pd.DataFrame({
    'Métriques': ['R² Score', 'RMSE', 'MAE'],
    'Entraînement': [train_r2, train_RMSE, train_MAE],
    'Test': [test_r2, test_RMSE, test_MAE]
})
print("\n Performance du modèle optimisé :")
print(performance_table)


 Performance du modèle optimisé :
  Métriques  Entraînement       Test
0  R² Score           1.0   0.446818
1      RMSE           0.0  55.307565
2       MAE           0.0  45.189599
